# Results  

This notebook creates files *candidates_all.fits*, and *candidates_best.fits*. The first contains a table with all candidates found at all plate pairs and sequences. It contains the summarized result of the entire project to date. The second contains the subset of candidates that, after visual inspection, look more promising as being real, and not artifacts that mimic stars in their integrated properties.

Final results from the pipeline are stored in subdirectory *plateanalysis/footprints/results/*. 

Each HTML file contains the result of a pipeline run over a *sequence* of plates. Sequences of plates are defined in file *settings.py*, and created based on the output of notebook *footprints.ipynb*.

HTML files contain just the very final product for each sequence; intermediate results are not stored in GitHub due to their policy of minimizing the storage of files that are either temporary, or potentially large, such as FITS binary tables and such. Intermediate results can be seen only by running the code locally over the appropriate data sets.  

#### Note

Due to sheer size, only one of the plate sequences I worked with has its pipeline output uploaded to the repo. Also, GitHub is unable to display the html due to its size. Download the file and use your browser locally to display the file.  

Alternatively, you can download the result files from Google Drive: https://drive.google.com/drive/folders/164xc4faMDbPt84ZileJZoHzK94kvG8Gm


Below is a summary of findings to date. Numbers refer to the plate ID code in the APPLAUSE archive.

- Data from the **1m-Spiegelteleskop** and the **Doppel-Reflektor** telescopes: still to be reviewed in light of the upgraded software.

- Data from the **Grosser Schmidt-Spiegel** telescope yeld a few candidates.




### Notes

APPLAUSE scans were performed on the original plates, not copies.

## Processing stats

Here, we build a summary of number of detections/objects for each plate pair, plus plate metadata information. The goal is to provide a bird's view of the coverage of the project to date, as well as a complete list of candidates in a single table.

In [1]:
import os
import warnings

import numpy as np

from astropy.table import Table, vstack, hstack
from astropy.time import Time
from astropy.utils.exceptions import AstropyWarning

import pandas as pd
from erfa import ErfaWarning

from library import get_earth_shadow
import settings
from settings import CATALOG, RESULTS, get_parameters, fname, sequences

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=ErfaWarning)
warnings.simplefilter('ignore', category=FutureWarning)

In [3]:
table_header = {
    'summary': {
        'names': ['seq', 'plate 1', 'ut_mid_1', 'exptime_1', 'es_1','plate 2', 'ut_mid_2', 
                  'exptime_2', 'es_2', 'total sources', 'matched', 'non-matched', 'candidates'],
        'dtypes': ['S1', 'S1', 'S1', 'i4', 'f4', 'S1', 'S1', 'i4', 'f4', 'i4', 'i4', 'i4', 'i4'],
    },
    'metadata': {
        'names': ['ut_mid', 'exptime', 'es', 'plate_id_next', 'ut_mid_next', 'exptime_next', 'es_next'],
        'dtypes': ['S1', 'i4', 'f4', 'i4', 'S1', 'i4', 'f4'],
    },
}

In [4]:
def make_table(table_type):
    table = Table(names=table_header[table_type]['names'],
                  dtype=table_header[table_type]['dtypes'])
    return table

In [5]:
# use Pandas to display table so all rows can be displayed at once. Displaying long
# Astropy tables directly causes only the first and last few rows to be displayed. 

def display_with_pandas(table):
    df = table.to_pandas()

    # avoid display of unicode reminder in every string field
    df['seq'] = df['seq'].apply(lambda x: x.decode('utf-8'))
    df['plate 1'] = df['plate 1'].apply(lambda x: x.decode('utf-8'))
    df['plate 2'] = df['plate 2'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_1'] = df['ut_mid_1'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_2'] = df['ut_mid_2'].apply(lambda x: x.decode('utf-8'))

    # style with CSS properties for table headers (th) and data cells (td)
    th_props = [('font-size', '12px'), ('text-align', 'right'), ('font-weight', 'bold')]
    td_props = [('font-size', '11px'), ('text-align', 'right')]
    styles = [dict(selector="th", props=th_props),dict(selector="td", props=td_props)]
    styled_df = df.style.set_table_styles(styles)    
    
    format_mapping = {
        'es_1': '{:.4}'.format,
        'es_2': '{:.4}'.format,
    }
    styled_df = styled_df.format(format_mapping)    

    display(styled_df)

In [6]:
# get metadata for a given plate. Metadata is taken from original input table from applause.

def get_metadata(plate, catalog_table):
    
    mask = catalog_table['plate_id'] == plate
    t = catalog_table[mask]
    
    long = t['site_longitude'][0]
    lat  = t['site_latitude'][0]
    ra  = t['ra_icrs'][0]
    dec = t['dec_icrs'][0]
    time_event = Time(t['ut_mid'][0])

    es, _ = get_earth_shadow(ra, dec, time_event, longitude=long, latitude=lat)
    
    return t['ut_mid'][0], t['exptime'][0], es

In [7]:
def add_metadata(metadata, table):
    
    # create temporary table to contain metadata
    meta_table = make_table('metadata')
    
    for row_index in range(len(table)):
        plate_id = table['plate_id_1'][row_index]

        plate_id_next = metadata[plate_id]['plate_id_next']    
        ut_mid        = metadata[plate_id]['ut_mid']    
        exptime       = metadata[plate_id]['exptime']    
        es            = metadata[plate_id]['es']    
        ut_mid_next   = metadata[plate_id]['ut_mid_next']    
        exptime_next  = metadata[plate_id]['exptime_next']    
        es_next       = metadata[plate_id]['es_next']    
    
        meta_table.add_row([ut_mid, exptime,es, plate_id_next, ut_mid_next, exptime_next, es_next])
  
    # join with main table
    result = hstack([meta_table, table])
    
    return result    

In [8]:
# metadata comes from original input table from applause
catalog_table = Table.read(CATALOG, format='ascii.csv')

# summary table
result = make_table('summary')

# list of tables to collect all candidate events
candidates_list = []

# collect some metadata here
metadata = {}

In [9]:
# 'sequences' is a dict defined in file settings.py

for seq_key in list(sequences.keys()):
    seq = sequences[seq_key]

    for i in range(len((seq)) - 1):
        try:
            plate_id = seq[i]
            next_plate_id = seq[i+1]
            
            # metadata
            ut_mid_1, exptime_1, es_1 = get_metadata(plate_id, catalog_table)
            ut_mid_2, exptime_2, es_2 = get_metadata(next_plate_id, catalog_table)
            
            metadata[plate_id] = {
                'ut_mid': ut_mid_1,
                'exptime': exptime_1,
                'es': es_1,
                'plate_id_next': next_plate_id,
                'ut_mid_next': ut_mid_2,
                'exptime_next': exptime_2,
                'es_next': es_2,
            }
            
            plate_id_str = str(plate_id)
            next_plate_id_str = str(next_plate_id)

            key = plate_id_str + ',' + next_plate_id_str
            par = get_parameters(key)

            # row count on each relevant product table
            table_sources = Table.read(fname(par['table1']), format='ascii.csv')
            mask = table_sources['scan_id'] == table_sources['scan_id'][0]
            table_sources = table_sources[mask]
            sources = len(table_sources)

            table_matched = Table.read(fname(par['table_matched']), format='fits')
            matched = len(table_matched)

            table_non_matched = Table.read(fname(par['table_non_matched']), format='fits')
            non_matched = len(table_non_matched)

            table_candidates = Table.read(fname(par['table_candidates']), format='fits')
            candidates = len(table_candidates)
            
            # add this candidates table to final product. Add seq_key as identifier
            candidates_list.append((seq_key, table_candidates))
            
            result.add_row([seq_key, plate_id_str, ut_mid_1, exptime_1, es_1,
                            next_plate_id_str, ut_mid_2, exptime_2, es_2,
                            sources, matched, non_matched, candidates])
        except KeyError:
            break

### Summary table

Table columns:

- **seq**: sequence ID generated by notebook *footprints.ipynb*
- **plate ...**: plate IDs defining the pair of plates
- **ut_mid ...**: UT mid for each plate
- **exptime ...**: EXPTIME for each plate
- **es ...**: Earth's shadow angle (deg) for each plate
- **total sources**: total number of sources in the APPLAUSE table of the first plate, for the *_x* scan direction
- **matched**: number of sources in the first plate that have a match in the second plate, after the first plate's source table gets prunned of bad detections (as per sextractor-generated parameters)
- **non-matched**: number of sources in the first plate (after it was cleaned up as above) that have **NO** match on the second plate
- **candidates**: number of candidates remaining after filtering is applyied (filtering based on Gaussian fitting, radial profile, circularity, and aperture photometry analysis)


In [10]:
display_with_pandas(result)

,seq,plate 1,ut_mid_1,exptime_1,es_1,plate 2,ut_mid_2,exptime_2,es_2,total sources,matched,non-matched,candidates
0,seq01,9245,1956-09-25 19:00:47,1800,69.35,9246,1956-09-25 19:43:40,1800,68.44,590001,246737,27726,1
1,seq01,9246,1956-09-25 19:43:40,1800,68.44,9247,1956-09-25 20:29:33,1800,67.58,465264,233033,16063,14
2,seq03,9313,1956-12-03 17:15:31,900,62.87,9315,1956-12-03 18:06:23,900,61.75,55239,17444,152,0
3,seq03,9315,1956-12-03 18:06:23,900,61.75,9317,1956-12-03 18:56:48,900,60.63,76057,22383,2274,0
4,seq03,9317,1956-12-03 18:56:48,900,60.63,9318,1956-12-03 19:26:11,900,59.99,59981,21741,488,0
5,seq03,9318,1956-12-03 19:26:11,900,59.99,9319,1956-12-03 20:27:18,900,58.8,62314,21577,200,1
6,seq03,9319,1956-12-03 20:27:18,900,58.8,9320,1956-12-03 20:55:56,900,58.28,75926,25112,455,1
7,seq04,9322,1956-12-06 18:54:56,900,61.95,9323,1956-12-06 19:19:52,900,61.4,58894,21021,367,1
8,seq04,9323,1956-12-06 19:19:52,900,61.4,9324,1956-12-06 19:44:24,900,60.91,55417,21597,543,0
9,seq04,9324,1956-12-06 19:44:24,900,60.91,9325,1956-12-06 20:08:14,900,60.41,54232,19220,148,1


In [11]:
print("Total sources:         ", np.sum(result['total sources']))
print("Total matched sources: ", np.sum(result['matched']))
print("Total non matched:     ", np.sum(result['non-matched']))
print("Total candidates:      ", np.sum(result['candidates']))

Total sources:          9294162
Total matched sources:  4495413
Total non matched:      263630
Total candidates:       246


## Final candidates tables

All candidates from all plate pairs are stored together into a single FITS table. Add some metadata to help navigate the table.

In [12]:
seq_list, table_list = map(list, zip(*candidates_list))

candidates_table = vstack(table_list)

candidates_table = add_metadata(metadata, candidates_table)

filename = os.path.join(RESULTS, "candidates_all.fits")
candidates_table.write(filename, overwrite=True)
print(len(candidates_table))

246


## Visual vetting

Unfortunately, the code is not yet capable of filtering out many detections that look suspect to the eye. In this section, we manually build the list of *source_id* values of detections that look acceptable. 

Tipically, the objects look round, and have radial profiles that closely match star profiles at least in shape, if not in FWHM.

Procedure for visual vetting:

- from the candidates table, print all sequence and source IDs;
- copy-paste the output into file *best.csv*;
- remove from this file, by hand, anything that looks weird;
- read table from edited file *best.csv*;
- use table's *source_id* colun to filter the candidates table;
- write result into *candidates_best.fits* file.

**Note - remember to preserve existing source IDs**. This is the primary storage place for these IDs, which are in fact the central result of the project.

In [44]:
# WE DON'T NEED THIS ONCE VISUAL VETTING HAPPENS, THAT IS, ONCE THE 'best.csv' FILE IS CREATED AND EDITED. 

# # This creates the input to build file 'best.csv'

# # reserve some columns for taking notes in the csv table
# print("sequence, source_id, , remove, notes1, notes2")
# for seq, table in zip(seq_list, table_list):
#     source_ids = table['source_id']
#     for sid in source_ids: 
#         print(seq + ", " + str(sid) + ", 0, ,")

Once 'best.csv' is edited by hand, build and write candidates table with the "best" detections.

In [43]:
table_best = Table.read(os.path.join(RESULTS, 'best.csv'), format='ascii.csv')

mask =(1 - table_best['remove']).astype(bool)
table_best = table_best[mask]

mask = np.isin(candidates_table['source_id'], table_best['source_id'])
candidates_table_best = candidates_table[mask]

filename = os.path.join(RESULTS, "candidates_best.fits")
candidates_table_best.write(filename, overwrite=True)
print(len(candidates_table_best))

93


## To do

- look for alignments.
- augment dataset with:
  - sequences with two plates only - DONE
  - sequences that reach over multiple nights.
- fix *footprints.ipynb*: time order in sequences of plates. This is relevant only for long sequences, so low priority for now.

- multivariate probability - NO NEED for now.
- add more shape diagnostics - DONE
- add circularity diagnostic - DONE
- profile metrics - DONE
- refactor towards batch processing - DONE
- automate data download - DONE
- add Earth shadow angle to tables - DONE

## Summary of possible bright event in plate 9319

### Needs revision: other similar events exist - do it after visual vetting

- date 1956-12-03
- time between utc_mid: 28 min
- exposure time on each plate: 15 min
- emulsion Kodak Oa-E both plates

Plates 9313, 9315, 9316, 9317, 9318, were searched for similar events, with no detection. These plates were all taken on the same night of 1956-12-03 over a 4 hr time span, with the same telescope pointing (except 9316 that was taken 24 hr later). 

The event position on the plate (distance to center = 3248.3 px, distance to edge = 1061.8 px, annular bin = 4) places it in a region where artifacts can be perhaps more likely to happen. The profile suggests that it might be indeed an artifact: these tend to have systematically narrower widths than stars of same brightness. Or, could be indeed the result of a very brief flash of light: these are argued to appear with somewaht narrower PSFs than long-exposure star images.

Plate's FOV is far from Earth's shadow. 

Comparing magnitudes in the neighborhood, we can estimate the transient has a blue magnitude of $\sim16.0$. Assuming the transient is fast ($\sim0.1$ s), it will have a magnitude of $\sim6.1$.
